In [ ]:
# ==============================================================================
# CELL 0: HARDWARE VERIFICATION & DEPENDENCY SETUP
# ==============================================================================
import subprocess
import sys
import os
import torch

print('=' * 65)
print('  UNSLOTH GEMMA 4 31B LORA TRAINING SETUP (NVIDIA L4 24GB)')
print('=' * 65)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available! Ensure GPU accelerator is enabled.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('Hardware does not support native bfloat16! Zero fp16 fallback allowed.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)')
assert vram_gb >= 20.0, f'Expected >= 20 GB VRAM (L4/A100), found {vram_gb:.2f} GB!'

print('\nInstalling Unsloth, TRL, PEFT, bitsandbytes, and compressed-tensors...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch', 'torchvision', 'torchaudio',
    'bitsandbytes', 'transformers', 'accelerate', 'trl', 'peft',
    'compressed-tensors', 'llm-compressor'
], check=True)

try:
    import unsloth
    print('Unsloth is already installed.')
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/unslothai/unsloth.git'
    ], check=True)
    import unsloth
    print('Unsloth installed successfully.')


In [ ]:
# ==============================================================================
# CELL 1: DATASET INGESTION & ZERO-THOUGHT PREFIX-DELTA VERIFICATION
# ==============================================================================
import json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
DATA_DIR = WORKING_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

candidate_train = (
    sorted(Path('/kaggle/input').rglob('unsloth_sft_train.jsonl')) +
    sorted(Path('/kaggle/working/data').rglob('unsloth_sft_train.jsonl')) +
    sorted(Path('./data').rglob('unsloth_sft_train.jsonl'))
)
assert candidate_train, 'Fatal: unsloth_sft_train.jsonl not found!'
train_path = candidate_train[0]
val_path = train_path.parent / 'unsloth_sft_val.jsonl'
assert val_path.exists(), f'Fatal: Missing validation dataset at {val_path}!'

for p in [train_path, val_path]:
    content = p.read_text(encoding='utf-8')
    lines = [json.loads(line) for line in content.splitlines() if line.strip()]
    for r in lines:
        assert r['text'].startswith(r['prefix_text'])
        assert r['completion_text'].startswith('<|tool_call>call:')
        assert '<|channel>thought' not in r['completion_text']
    print(f'Verified {p.name}: {len(lines)} zero-thought samples, {len(content)} bytes')


In [ ]:
# ==============================================================================
# CELL 2: MODEL INITIALIZATION & RANK-8 PEFT CONFIGURATION
# ==============================================================================
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from unsloth import FastLanguageModel

max_seq_length = 3072
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='fp4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

def is_pack_quantized_candidate(cand: str) -> bool:
    if 'w4a16-ct' in cand.lower():
        return True
    cfg_file = Path(cand) / 'config.json'
    if cfg_file.exists():
        try:
            cfg = json.loads(cfg_file.read_text(encoding='utf-8'))
            q_cfg = cfg.get('quantization_config', {})
            if q_cfg.get('format') == 'pack-quantized' or q_cfg.get('quant_method') == 'compressed-tensors':
                return True
        except Exception:
            pass
    return False

candidate_models = [
    'google/gemma-4-31b-it-qat-w4a16-ct',
    'google/gemma-4-31B-it-qat-q4_0-unquantized',
]

model = None
tokenizer = None
for cand in candidate_models:
    if 'bnb-4bit' in cand.lower():
        continue
    is_pack_quant = is_pack_quantized_candidate(cand)
    print(f'Attempting to load model from: {cand} (pack_quantized={is_pack_quant})...')
    try:
        if is_pack_quant:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=cand,
                max_seq_length=max_seq_length,
                dtype=torch.bfloat16,
                load_in_4bit=False,
                use_exact_model_name=True,
            )
        else:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=cand,
                max_seq_length=max_seq_length,
                dtype=torch.bfloat16,
                load_in_4bit=True,
                quantization_config=bnb_config,
                use_exact_model_name=True,
            )
        if tokenizer is None:
            tokenizer = AutoTokenizer.from_pretrained(cand)
        print(f'Successfully loaded {cand}!')
        break
    except Exception as exc:
        print(f'Failed loading {cand}: {exc}')
        continue

assert model is not None and tokenizer is not None, 'Fatal: Could not load any candidate Gemma 4 model!'
assert len(tokenizer) == 262144, f'Expected 262144 vocab size, got {len(tokenizer)}'

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'v_proj', 'o_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
)
model.print_trainable_parameters()


In [ ]:
# ==============================================================================
# CELL 3: PRE-TOKENIZED PREFIX-DELTA SFT TRAINING (NO LOGIT OOM)
# ==============================================================================
import torch
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

raw_train = load_dataset('json', data_files=str(train_path), split='train')

def tokenize_with_prefix_mask(example):
    p_ids = tokenizer(example['prefix_text'], add_special_tokens=False)['input_ids']
    f_ids = tokenizer(example['text'], truncation=True, max_length=max_seq_length, add_special_tokens=False)['input_ids']
    mask_len = min(len(p_ids), len(f_ids))
    labels = [-100] * mask_len + list(f_ids[mask_len:])
    return {'input_ids': f_ids, 'attention_mask': [1] * len(f_ids), 'labels': labels}

train_dataset = raw_train.map(tokenize_with_prefix_mask, remove_columns=raw_train.column_names)

class PrefixDeltaCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id
    def __call__(self, features):
        max_len = max(len(f['input_ids']) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad_len = max_len - len(f['input_ids'])
            input_ids.append(f['input_ids'] + [self.pad_token_id] * pad_len)
            attention_mask.append(f['attention_mask'] + [0] * pad_len)
            labels.append(f['labels'] + [-100] * pad_len)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
        }

OUTPUT_DIR = WORKING_DIR / 'training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=15,
    num_train_epochs=1,
    learning_rate=1.5e-4,
    fp16=False,
    bf16=True,
    logging_steps=5,
    eval_strategy='no',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type='cosine',
    neftune_noise_alpha=None,
    seed=3407,
    report_to='none',
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    data_collator=PrefixDeltaCollator(tokenizer.pad_token_id or 0),
    args=training_args,
)

train_stats = trainer.train()
print(train_stats)


In [ ]:
# ==============================================================================
# CELL 4: EXPORT & NORMALIZE LORA ADAPTER WEIGHTS (STRICTLY 2 FILES FOR VLLM)
# ==============================================================================
import json
import os
from pathlib import Path
import torch
from safetensors.torch import load_file, save_file

def normalize_adapter_for_vllm(adapter_dir: Path) -> None:
    adapter_dir = Path(adapter_dir)
    weights_path = adapter_dir / 'adapter_model.safetensors'
    config_path = adapter_dir / 'adapter_config.json'
    assert weights_path.exists(), f'Missing safetensors weights at {weights_path}'
    assert config_path.exists(), f'Missing adapter config at {config_path}'

    target_prefix = 'base_model.model.language_model.model.layers.'
    tensors = load_file(str(weights_path))
    normalized_tensors = {}
    for key, val in tensors.items():
        new_key = key
        if 'base_model.model.model.layers.' in key and 'language_model.' not in key:
            new_key = key.replace('base_model.model.model.layers.', target_prefix)
        elif 'base_model.model.layers.' in key and 'language_model.' not in key:
            new_key = key.replace('base_model.model.layers.', target_prefix)
        if val.dtype != torch.bfloat16:
            val = val.to(torch.bfloat16)
        normalized_tensors[new_key] = val

    config = json.loads(config_path.read_text(encoding='utf-8'))
    config['target_modules'] = ['o_proj', 'q_proj', 'v_proj']
    config['r'] = 8
    config['lora_alpha'] = 16

    save_file(normalized_tensors, str(weights_path), metadata={'format': 'pt'})
    config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')

    for p in list(adapter_dir.iterdir()):
        if p.is_file() and p.name not in ('adapter_config.json', 'adapter_model.safetensors'):
            p.unlink()

    assert len(normalized_tensors) == 340, f'Expected 340 tensors, got {len(normalized_tensors)}'
    for name, tensor in normalized_tensors.items():
        assert name.startswith(target_prefix), f'Key {name} missing vLLM prefix {target_prefix}'
        assert tensor.dtype == torch.bfloat16, f'Key {name} has dtype {tensor.dtype}, expected bfloat16'
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)
    assert size_mb <= 35.0, f'Adapter file size {size_mb:.2f} MB exceeds 35.0 MB limit'
    print(f'Normalized vLLM adapter: 340 BF16 tensors ({size_mb:.2f} MB).')

ADAPTER_DIR = WORKING_DIR / 'submission' / 'adapters' / 'main_lora'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR))
normalize_adapter_for_vllm(ADAPTER_DIR)

assert (ADAPTER_DIR / 'adapter_config.json').exists(), 'Missing adapter_config.json!'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), 'Missing adapter_model.safetensors!'

CONFIGS_DIR = WORKING_DIR / 'submission' / 'configs'
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
sampling_yaml = """temperature: 0.2
max_output_tokens: 16384
thinking_config:
  include_thoughts: false
  thinking_budget: 0
"""
(CONFIGS_DIR / 'sampling.yaml').write_text(sampling_yaml, encoding='utf-8')
print('Exported normalized adapter_config.json + adapter_model.safetensors and zero-thought sampling.yaml.')
